In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter

def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):

    # box_format이 midpoint일 경우 중심 좌표와 크기를 바탕으로 좌표 계산
    if box_format == "midpoint":
        # 예측 박스 좌상단(x1, y1), 우하단(x2, y2) 계산
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2

        # 정답 박스 좌상단(x1, y1), 우하단(x2, y2) 계산
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    # box_format이 corners일 경우 이미 (x1, y1, x2, y2) 형식이므로 그대로 사용
    if box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    # 두 박스 간 겹치는 영역의 좌상단, 우하단 좌표 계산
    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # 교차 영역의 넓이 계산. 교차하지 않는 경우 clamp(0)으로 음수 제거
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    # 각 박스의 전체 면적 계산
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    return intersection / (box1_area + box2_area - intersection + 1e-6)


def non_max_suppression(bboxes, iou_threshold, threshold, box_format="corners"):

    assert type(bboxes) == list 

    bboxes = [box for box in bboxes if box[1] > threshold]

    # confidence score를 기준으로 내림차순 정렬
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)

    # NMS 후 남을 박스를 저장할 리스트
    bboxes_after_nms = []

    # 남은 박스가 있을 때까지 반복
    while bboxes:
        # 가장 confidence score가 높은 박스를 선택
        chosen_box = bboxes.pop(0)

        # 선택한 박스와 비교하여 중복되는 박스들 제거
        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]  # 클래스가 다르면 유지
            or intersection_over_union(
                torch.tensor(chosen_box[2:]),  # [x1, y1, x2, y2] or [cx, cy, w, h]
                torch.tensor(box[2:]),
                box_format=box_format,
            ) < iou_threshold  # IoU가 threshold 미만이면 유지
        ]

        # 중복되지 않는 박스를 최종 리스트에 추가
        bboxes_after_nms.append(chosen_box)

    return bboxes_after_nms


def mean_average_precision(
    pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint", num_classes=20
):

    average_precisions = []  # 각 클래스별 average precision 저장 리스트
    epsilon = 1e-6  # 나눗셈에서 0으로 나누는 것을 방지하기 위한 작은 값

    # 클래스별로 반복
    for c in range(num_classes):
        detections = []   # 현재 클래스의 예측 박스들
        ground_truths = []  # 현재 클래스의 실제 박스들

        # 현재 클래스에 해당하는 예측 박스만 필터링
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)

        # 현재 클래스에 해당하는 실제 박스만 필터링
        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)

        # 이미지별로 ground truth 박스 수를 계산
        amount_bboxes = Counter([gt[0] for gt in ground_truths])

        # 이미지별로 각 GT 박스가 이미 매칭되었는지 확인할 tensor 초기화
        for key, val in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)  # 각 이미지에 대해 [0, 0, ..., 0]

        # confidence score를 기준으로 예측 박스들을 내림차순 정렬
        detections.sort(key=lambda x: x[2], reverse=True)

        TP = torch.zeros((len(detections)))  # True Positive 배열
        FP = torch.zeros((len(detections)))  # False Positive 배열
        total_true_bboxes = len(ground_truths)  # 현재 클래스의 GT 총 개수

        # GT가 없으면 해당 클래스는 스킵
        if total_true_bboxes == 0:
            continue

        # 모든 detection에 대해 GT와 비교하여 TP/FP 판별
        for detection_idx, detection in enumerate(detections):
            # detection이 속한 이미지의 GT 박스만 추출
            ground_truth_img = [
                bbox for bbox in ground_truths if bbox[0] == detection[0]
            ]

            best_iou = 0  # 현재 detection에 대해 최고 IoU
            best_gt_idx = -1  # 가장 IoU가 높은 GT의 인덱스

            # GT 박스들과 IoU 비교
            for idx, gt in enumerate(ground_truth_img):
                iou = intersection_over_union(
                    torch.tensor(detection[3:]),  # 예측 박스 좌표
                    torch.tensor(gt[3:]),         # GT 박스 좌표
                    box_format=box_format,
                )

                # 가장 높은 IoU 업데이트
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            # IoU가 threshold보다 크면 TP로 간주할 수 있음
            if best_iou > iou_threshold:
                # 해당 GT가 아직 매칭되지 않았다면 TP
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1  # 매칭됨 표시
                else:
                    # 이미 매칭된 GT와 중복되는 예측이면 FP
                    FP[detection_idx] = 1
            else:
                # IoU 기준 미달 → FP
                FP[detection_idx] = 1

        # 누적 TP/FP 계산 (precision-recall curve 계산용)
        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(FP, dim=0)

        # recall = 누적 TP / 전체 GT 개수
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        # precision = 누적 TP / (누적 TP + 누적 FP)
        precisions = torch.divide(TP_cumsum, (TP_cumsum + FP_cumsum + epsilon))

        # precision과 recall 앞에 시작점 추가 (0에서 시작)
        precisions = torch.cat((torch.tensor([1]), precisions))
        recalls = torch.cat((torch.tensor([0]), recalls))

        # precision-recall curve 아래 넓이를 적분하여 AP 계산
        average_precisions.append(torch.trapz(precisions, recalls))

    # mAP = 클래스별 AP 평균
    return sum(average_precisions) / len(average_precisions)


def plot_image(image, boxes):

    # 이미지를 numpy 배열로 변환
    im = np.array(image)
    height, width, _ = im.shape  # 이미지의 높이와 너비 추출

    # 플롯 생성
    fig, ax = plt.subplots(1)
    ax.imshow(im)  # 이미지 출력

    # 박스 시각화: [x_center, y_center, width, height] 형식
    for box in boxes:
        box = box[2:]  # class_pred, prob_score는 무시하고 [x, y, w, h]만 사용

        # 박스 길이가 정확히 4가 아니면 오류
        assert len(box) == 4, "Got more values than in x, y, w, h, in a box!"

        # 중심 좌표 → 좌상단 좌표로 변환
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2

        # matplotlib의 Rectangle은 좌상단 좌표 기준이므로 변환된 좌표와 크기 사용
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),  # 좌상단 좌표
            box[2] * width,   # width
            box[3] * height,  # height
            linewidth=1,
            edgecolor="r",    # 빨간 테두리
            facecolor="none"  # 내부는 투명
        )

        # Axes에 박스 추가
        ax.add_patch(rect)

    # 결과 시각화 출력
    plt.show()


def get_bboxes(
    loader,
    model,
    iou_threshold,
    threshold,
    pred_format="cells",  # 사용되지 않지만 cell-based 예측임을 암시
    box_format="midpoint",
    device="cuda",
):

    all_pred_boxes = []  # 모든 이미지의 예측 박스 저장
    all_true_boxes = []  # 모든 이미지의 실제 박스 저장

    model.eval()  # 모델을 평가 모드로 전환 (dropout, batchnorm 비활성)
    train_idx = 0  # 각 이미지의 인덱스를 추적하기 위한 카운터

    for batch_idx, (x, labels) in enumerate(loader):
        x = x.to(device)          # 이미지 배치를 장치에 올림
        labels = labels.to(device)  # 레이블도 장치에 올림

        with torch.no_grad():  # 평가 중이므로 그래디언트 계산은 비활성화
            predictions = model(x)  # 모델 예측 수행

        batch_size = x.shape[0]

        # 예측 및 실제 박스를 셀 기반 YOLO output에서 일반적인 박스 형식으로 변환
        true_bboxes = cellboxes_to_boxes(labels)      # shape: [B, N, 6 or 7]
        bboxes = cellboxes_to_boxes(predictions)

        for idx in range(batch_size):
            # 한 이미지에 대해 NMS 적용
            nms_boxes = non_max_suppression(
                bboxes[idx],                     # 해당 이미지의 예측 박스 리스트
                iou_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )

            # 이미지와 연관된 예측 박스를 저장 (image index + box 정보)
            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            # 이미지와 연관된 정답 박스를 저장 (threshold 이상만 사용)
            for box in true_bboxes[idx]:
                if box[1] > threshold:  # confidence가 threshold보다 큰 경우만 유지
                    all_true_boxes.append([train_idx] + box)

            train_idx += 1  # 다음 이미지 인덱스로 증가

    model.train()  # 모델을 다시 학습 모드로 되돌림
    return all_pred_boxes, all_true_boxes  # 모든 예측/정답 박스 반환




def convert_cellboxes(predictions, S=7):

    predictions = predictions.to("cpu")  # GPU -> CPU
    batch_size = predictions.shape[0]

    # (B, 7, 7, 30)로 reshape
    predictions = predictions.reshape(batch_size, 7, 7, 30)

    # 두 개의 bounding box 추출 (YOLO는 B=2)
    bboxes1 = predictions[..., 21:25]  # 첫 번째 box (x, y, w, h)
    bboxes2 = predictions[..., 26:30]  # 두 번째 box

    # confidence score 비교하여 더 높은 박스 선택
    scores = torch.cat(
        (predictions[..., 20].unsqueeze(0), predictions[..., 25].unsqueeze(0)), dim=0
    )
    best_box = scores.argmax(0).unsqueeze(-1)  # 두 박스 중 더 높은 confidence index 선택

    # 높은 confidence의 박스만 선택
    best_boxes = bboxes1 * (1 - best_box) + best_box * bboxes2

    # 셀의 x좌표: (0 ~ 6) → (batch, 7, 7, 1)
    cell_indices = torch.arange(7).repeat(batch_size, 7, 1).unsqueeze(-1)

    # 전체 이미지 기준 x, y 좌표 (셀 위치 + 상대 좌표) * 1/S
    x = 1 / S * (best_boxes[..., :1] + cell_indices)
    y = 1 / S * (best_boxes[..., 1:2] + cell_indices.permute(0, 2, 1, 3))

    # 너비, 높이도 1/S로 정규화
    w_y = 1 / S * best_boxes[..., 2:4]

    # 최종 좌표 구성: (x, y, w, h)
    converted_bboxes = torch.cat((x, y, w_y), dim=-1)

    # 클래스 예측 (one-hot 벡터 중 argmax)
    predicted_class = predictions[..., :20].argmax(-1).unsqueeze(-1)

    # 선택된 박스의 confidence score
    best_confidence = torch.max(predictions[..., 20], predictions[..., 25]).unsqueeze(-1)

    # 최종 output: (class, confidence, x, y, w, h)
    converted_preds = torch.cat(
        (predicted_class, best_confidence, converted_bboxes), dim=-1
    )

    return converted_preds  # shape: (B, 7, 7, 6)


def cellboxes_to_boxes(out, S=7):

    # 각 셀의 박스를 (B, S*S, 6) 형식으로 reshape
    converted_pred = convert_cellboxes(out).reshape(out.shape[0], S * S, -1)

    # 클래스 인덱스를 정수형으로 변환
    converted_pred[..., 0] = converted_pred[..., 0].long()

    all_bboxes = []

    # 각 배치별로 박스를 리스트에 저장
    for ex_idx in range(out.shape[0]):
        bboxes = []
        for bbox_idx in range(S * S):
            # tensor를 Python list로 변환
            bboxes.append([x.item() for x in converted_pred[ex_idx, bbox_idx, :]])
        all_bboxes.append(bboxes)

    return all_bboxes


def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    
    print("=> Saving checkpoint")
    torch.save(state, filename)



def load_checkpoint(checkpoint, model, optimizer):
   
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

In [ ]:

import torch
import os
import pandas as pd
from PIL import Image


class VOCDataset(torch.utils.data.Dataset):
    def __init__(
        self, csv_file, img_dir, label_dir, S=7, B=2, C=20, transform=None,
    ):
        """
        Parameters:
            csv_file (str): 이미지 파일명과 라벨 파일명이 들어 있는 CSV 경로
            img_dir (str): 이미지 디렉토리 경로
            label_dir (str): 라벨(.txt) 파일이 있는 디렉토리 경로
            S (int): YOLO에서 사용되는 grid 개수 (기본 7x7)
            B (int): 각 cell당 예측하는 bounding box 개수 (기본 2개)
            C (int): 클래스 개수 (PASCAL VOC는 20개)
            transform (callable): 이미지 및 박스에 적용할 전처리 함수
        """
        self.annotations = pd.read_csv(csv_file)  # CSV에서 (img, label) 정보 읽기
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.S = S
        self.B = B
        self.C = C

    def __len__(self):
        # 전체 데이터 수 (이미지 수) 반환
        return len(self.annotations)

    def __getitem__(self, index):
        # 라벨 파일 경로 설정
        label_path = os.path.join(self.label_dir, self.annotations.iloc[index, 1])
        boxes = []

        # 라벨 파일 읽기
        with open(label_path) as f:
            for label in f.readlines():
                # class, x_center, y_center, width, height
                class_label, x, y, width, height = [
                    float(x) if float(x) != int(float(x)) else int(x)
                    for x in label.replace("\n", "").split()
                ]
                boxes.append([class_label, x, y, width, height])  # 리스트로 저장

        # 이미지 경로 설정 및 열기
        img_path = os.path.join(self.img_dir, self.annotations.iloc[index, 0])
        image = Image.open(img_path)
        boxes = torch.tensor(boxes)

        # 전처리(transform) 적용
        if self.transform:
            image, boxes = self.transform(image, boxes)

        # YOLO 학습을 위한 label_matrix 생성: (S, S, C + 5*B)
        # 각 cell에서: C(클래스 one-hot) + 5(B박스) = 25
        label_matrix = torch.zeros((self.S, self.S, self.C + 5 * self.B))

        for box in boxes:
            class_label, x, y, width, height = box.tolist()
            class_label = int(class_label)

            # (x, y)가 속한 cell의 인덱스 i(row), j(column) 계산
            i, j = int(self.S * y), int(self.S * x)

            # cell 안에서의 상대 좌표 계산
            x_cell = self.S * x - j  # x의 소수점만 남김
            y_cell = self.S * y - i

            # 셀 크기에 맞게 width/height 조정
            width_cell = width * self.S
            height_cell = height * self.S

            # 해당 셀에 객체가 아직 할당되지 않았을 때만 처리
            if label_matrix[i, j, 20] == 0:
                label_matrix[i, j, 20] = 1  # objectness score = 1

                # 좌표 저장 (B=2이지만 여기선 첫 박스만 씀)
                box_coordinates = torch.tensor(
                    [x_cell, y_cell, width_cell, height_cell]
                )
                label_matrix[i, j, 21:25] = box_coordinates

                # 클래스 레이블 one-hot 인코딩
                label_matrix[i, j, class_label] = 1

        return image, label_matrix  # 이미지와 YOLO용 라벨 반환

In [ ]:
"""
YOLO (v1) 모델 구현 - BatchNorm 추가된 버전
"""

import torch
import torch.nn as nn

# YOLO 아키텍처 설정 (YOLO v1 구조 기반)
# 튜플: (kernel_size, out_channels, stride, padding)
# "M": MaxPooling (kernel=2, stride=2)
# 리스트: [conv1, conv2, 반복 횟수]
architecture_config = [
    (7, 64, 2, 3),
    "M",
    (3, 192, 1, 1),
    "M",
    (1, 128, 1, 0),
    (3, 256, 1, 1),
    (1, 256, 1, 0),
    (3, 512, 1, 1),
    "M",
    [(1, 256, 1, 0), (3, 512, 1, 1), 4],
    (1, 512, 1, 0),
    (3, 1024, 1, 1),
    "M",
    [(1, 512, 1, 0), (3, 1024, 1, 1), 2],
    (3, 1024, 1, 1),
    (3, 1024, 2, 1),
    (3, 1024, 1, 1),
    (3, 1024, 1, 1),
]



class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, bias=False, **kwargs)  # bias=False → BatchNorm으로 대체
        self.batchnorm = nn.BatchNorm2d(out_channels)  # BatchNorm으로 학습 안정화
        self.leakyrelu = nn.LeakyReLU(0.1)  # 활성화 함수

    def forward(self, x):
        return self.leakyrelu(self.batchnorm(self.conv(x)))  # 순서: Conv → BN → ReLU


class Yolov1(nn.Module):
    def __init__(self, in_channels=3, **kwargs):
        super(Yolov1, self).__init__()
        self.architecture = architecture_config
        self.in_channels = in_channels

        # CNN backbone 생성
        self.darknet = self._create_conv_layers(self.architecture)

        # FC layer 생성 (split_size, num_boxes, num_classes 사용)
        self.fcs = self._create_fcs(**kwargs)

    def forward(self, x):
        x = self.darknet(x)  # CNN 통과
        return self.fcs(torch.flatten(x, start_dim=1))  # FC 레이어 통과

    def _create_conv_layers(self, architecture):
        """
        아키텍처 정의에 따라 CNN 블록들을 구성하는 함수
        """
        layers = []
        in_channels = self.in_channels

        for x in architecture:
            if type(x) == tuple:
                # 일반 Conv block
                layers += [
                    CNNBlock(
                        in_channels, x[1], kernel_size=x[0], stride=x[2], padding=x[3],
                    )
                ]
                in_channels = x[1]  # 다음 레이어의 입력 채널 설정

            elif type(x) == str:
                # MaxPooling
                layers += [nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))]

            elif type(x) == list:
                # 반복되는 블록 처리
                conv1 = x[0]
                conv2 = x[1]
                num_repeats = x[2]

                for _ in range(num_repeats):
                    layers += [
                        CNNBlock(
                            in_channels,
                            conv1[1],
                            kernel_size=conv1[0],
                            stride=conv1[2],
                            padding=conv1[3],
                        )
                    ]
                    layers += [
                        CNNBlock(
                            conv1[1],
                            conv2[1],
                            kernel_size=conv2[0],
                            stride=conv2[2],
                            padding=conv2[3],
                        )
                    ]
                    in_channels = conv2[1]  # 반복 후 채널 업데이트

        return nn.Sequential(*layers)  # Sequential로 묶어 반환


    def _create_fcs(self, split_size, num_boxes, num_classes):

        S, B, C = split_size, num_boxes, num_classes

        # output 크기 = S * S * (C + B*5)
        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024 * S * S, 496),       # 원래는 4096, 여기선 496으로 축소 (메모리 절약 목적)
            nn.Dropout(0.0),
            nn.LeakyReLU(0.1),
            nn.Linear(496, S * S * (C + B * 5)),  # 각 셀당 클래스 + 2박스(xywh + confidence)
        )

In [ ]:
import torch
import torch.nn as nn


class YoloLoss(nn.Module):
    def __init__(self, S=7, B=2, C=20):

        super(YoloLoss, self).__init__()
        self.mse = nn.MSELoss(reduction="sum")  # 논문에서는 sum 사용

        self.S = S
        self.B = B
        self.C = C

        # 논문에서 제안한 가중치 값들
        self.lambda_noobj = 0.5   # 객체가 없을 때의 손실 비중
        self.lambda_coord = 5     # 박스 좌표 손실 비중


    def forward(self, predictions, target):
        # 입력 predictions: (BATCH_SIZE, S*S*(C + B*5)) 형태
        # → (B, S, S, 30) 로 reshape
        predictions = predictions.reshape(-1, self.S, self.S, self.C + self.B * 5)

        # 두 박스 각각의 IoU 계산
        iou_b1 = intersection_over_union(predictions[..., 21:25], target[..., 21:25])
        iou_b2 = intersection_over_union(predictions[..., 26:30], target[..., 21:25])
        ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)], dim=0)

        # 둘 중 더 높은 IoU를 갖는 박스를 선택
        iou_maxes, bestbox = torch.max(ious, dim=0)  # bestbox: 0 또는 1 인덱스
        exists_box = target[..., 20].unsqueeze(3)    # object가 존재하는지 여부 Iobj_i

        # ======================== #
        #   FOR BOX COORDINATES    #
        # ======================== #


        # 예측 중 더 좋은 박스를 선택하여 해당 좌표만 사용
        box_predictions = exists_box * (
            bestbox * predictions[..., 26:30] +
            (1 - bestbox) * predictions[..., 21:25]
        )

        box_targets = exists_box * target[..., 21:25]

        # 너비/높이에 루트를 씌워서 작은 박스에 더 집중하게 함 (논문 방식)
        box_predictions[..., 2:4] = torch.sign(box_predictions[..., 2:4]) * torch.sqrt(
            torch.abs(box_predictions[..., 2:4] + 1e-6)
        )
        box_targets[..., 2:4] = torch.sqrt(box_targets[..., 2:4])

        # 좌표 손실 (x, y, w, h)
        box_loss = self.mse(
            torch.flatten(box_predictions, end_dim=-2),
            torch.flatten(box_targets, end_dim=-2),
        )

        # ==================== #
        #   FOR OBJECT LOSS    #
        # ==================== #


        # 선택된 박스의 confidence score
        pred_box = bestbox * predictions[..., 25:26] + (1 - bestbox) * predictions[..., 20:21]

        object_loss = self.mse(
            torch.flatten(exists_box * pred_box),
            torch.flatten(exists_box * target[..., 20:21]),
        )

        # ======================= #
        #   FOR NO OBJECT LOSS    #
        # ======================= #

        # 객체가 없는 셀에 대해서는 두 박스 모두 confidence가 낮아야 함
        no_object_loss = self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 20:21], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1),
        )

        no_object_loss += self.mse(
            torch.flatten((1 - exists_box) * predictions[..., 25:26], start_dim=1),
            torch.flatten((1 - exists_box) * target[..., 20:21], start_dim=1),
        )

        # ================== #
        #   FOR CLASS LOSS   #
        # ================== #

        class_loss = self.mse(
            torch.flatten(exists_box * predictions[..., :20], end_dim=-2,),
            torch.flatten(exists_box * target[..., :20], end_dim=-2,),
        )

        loss = (
            self.lambda_coord * box_loss       # 좌표 손실 (가중치 5)
            + object_loss                      # 객체 confidence 손실
            + self.lambda_noobj * no_object_loss  # 객체 없음 손실 (가중치 0.5)
            + class_loss                       # 클래스 예측 손실
        )

        return loss

여기서부터는 데이터셋 구성

In [14]:
# VOC 2007 train+val
!wget http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
# VOC 2007 test
!wget http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar
# VOC 2012 train+val
!wget http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar

--2025-06-21 23:32:37--  http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
Resolving host.robots.ox.ac.uk (host.robots.ox.ac.uk)... 129.67.94.152
Connecting to host.robots.ox.ac.uk (host.robots.ox.ac.uk)|129.67.94.152|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 460032000 (439M) [application/x-tar]
Saving to: ‘VOCtrainval_06-Nov-2007.tar’

VOCtrainval_06-Nov- 100%[===================>] 438.72M  23.1MB/s    in 19s     

2025-06-21 23:32:57 (22.7 MB/s) - ‘VOCtrainval_06-Nov-2007.tar’ saved [460032000/460032000]

--2025-06-21 23:32:57--  http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar
Resolving host.robots.ox.ac.uk (host.robots.ox.ac.uk)... 129.67.94.152
Connecting to host.robots.ox.ac.uk (host.robots.ox.ac.uk)|129.67.94.152|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 451020800 (430M) [application/x-tar]
Saving to: ‘VOCtest_06-Nov-2007.tar’

VOCtest_06-Nov-2007 100%[===================

In [15]:
!tar -xf VOCtrainval_06-Nov-2007.tar
!tar -xf VOCtest_06-Nov-2007.tar
!tar -xf VOCtrainval_11-May-2012.tar

In [17]:
import xml.etree.ElementTree as ET
import os
from os import getcwd

sets = ['train', 'val', 'test']
classes = ["aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair",
           "cow", "diningtable", "dog", "horse", "motorbike", "person", "pottedplant",
           "sheep", "sofa", "train", "tvmonitor"]

def convert(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0 - 1
    y = (box[2] + box[3]) / 2.0 - 1
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

def convert_annotation(year, image_id):
    in_file = open(f'VOCdevkit/VOC{year}/Annotations/{image_id}.xml')
    out_file = open(f'VOCdevkit/VOC{year}/labels/{image_id}.txt', 'w')
    tree = ET.parse(in_file)
    root = tree.getroot()
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)

    for obj in root.iter('object'):
        difficult = obj.find('difficult').text
        cls = obj.find('name').text
        if cls not in classes or int(difficult) == 1:
            continue
        cls_id = classes.index(cls)
        xmlbox = obj.find('bndbox')
        b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
             float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
        bb = convert((w, h), b)
        out_file.write(f"{cls_id} " + " ".join([str(a) for a in bb]) + '\n')

def create_label_dir(year):
    label_path = f"VOCdevkit/VOC{year}/labels"
    if not os.path.exists(label_path):
        os.makedirs(label_path)

wd = getcwd()

for year, image_set in [('2007', 'train'), ('2007', 'val'), ('2007', 'test'), ('2012', 'train'), ('2012', 'val')]:
    create_label_dir(year)
    image_ids = open(f'VOCdevkit/VOC{year}/ImageSets/Main/{image_set}.txt').read().strip().split()
    list_file = open(f'{year}_{image_set}.txt', 'w')
    for image_id in image_ids:
        list_file.write(f'VOCdevkit/VOC{year}/JPEGImages/{image_id}.jpg\n')
        convert_annotation(year, image_id)
    list_file.close()

In [18]:
!cat 2007_train.txt 2007_val.txt 2012_*.txt > train.txt
!cp 2007_test.txt test.txt

In [19]:
!mkdir -p data/images
!mkdir -p data/labels

# 이미지 복사
!cp VOCdevkit/VOC2007/JPEGImages/*.jpg data/images/
!cp VOCdevkit/VOC2012/JPEGImages/*.jpg data/images/

# 라벨 복사
!cp VOCdevkit/VOC2007/labels/*.txt data/labels/
!cp VOCdevkit/VOC2012/labels/*.txt data/labels/

In [20]:
# generate_csv.py
import csv

def make_csv(txt_path, csv_path):
    with open(txt_path, "r") as f:
        lines = f.readlines()

    with open(csv_path, "w", newline="") as file:
        writer = csv.writer(file)
        for line in lines:
            image_file = line.strip().split("/")[-1]
            text_file = image_file.replace(".jpg", ".txt")
            writer.writerow([image_file, text_file])

make_csv("train.txt", "train.csv")
make_csv("test.txt", "test.csv")

학습 시작

In [33]:
"""
YOLOv1을 Pascal VOC 데이터셋에 대해 학습시키는 메인 스크립트
"""

import torch
import torchvision.transforms as transforms
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import DataLoader

# 랜덤 시드 고정 (재현성)
seed = 123
torch.manual_seed(seed)

# 하이퍼파라미터 및 설정
LEARNING_RATE = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16             # 논문에서는 64 사용. VRAM 부족 시 낮춤
WEIGHT_DECAY = 0
EPOCHS = 50
NUM_WORKERS = 2             # DataLoader에 사용할 subprocess 수
PIN_MEMORY = True           # GPU에 데이터를 빠르게 로드하기 위한 옵션
LOAD_MODEL = False          # 모델 로드 여부
LOAD_MODEL_FILE = "overfit.pth.tar"  # 저장된 모델 파일 경로
IMG_DIR = "data/images"     # 이미지 디렉터리
LABEL_DIR = "data/labels"   # 라벨 디렉터리


class Compose(object):
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, img, bboxes):
        for t in self.transforms:
            img, bboxes = t(img), bboxes # transforms는 이미지에만 적용

        return img, bboxes

# YOLOv1 입력 크기(448x448)로 resize하고, 텐서로 변환
transform = Compose([transforms.Resize((448, 448)), transforms.ToTensor(),])


def train_fn(train_loader, model, optimizer, loss_fn):
    loop = tqdm(train_loader, leave=True)  # tqdm으로 학습 진행 표시
    mean_loss = []

    for batch_idx, (x, y) in enumerate(loop):
        x, y = x.to(DEVICE), y.to(DEVICE)

        out = model(x)               # 모델 forward
        loss = loss_fn(out, y)       # 손실 계산
        mean_loss.append(loss.item())

        optimizer.zero_grad()        # 이전 gradient 초기화
        loss.backward()              # 역전파
        optimizer.step()             # 가중치 업데이트

        loop.set_postfix(loss=loss.item())  # 진행 바에 현재 손실 표시

    print(f"Mean loss was {sum(mean_loss)/len(mean_loss)}")

def main():
    # 모델 초기화 (YOLOv1)
    model = Yolov1(split_size=7, num_boxes=2, num_classes=20).to(DEVICE)
    # 옵티마이저 설정 (Adam 사용)
    optimizer = optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loss_fn = YoloLoss() # 손실 함수 정의

    # 저장된 모델 불러오기 (선택)
    if LOAD_MODEL:
        load_checkpoint(torch.load(LOAD_MODEL_FILE), model, optimizer)

    # 학습용 데이터셋 설정
    train_dataset = VOCDataset(
        "/content/100examples_yolo.csv",
        transform=transform,
        img_dir=IMG_DIR,
        label_dir=LABEL_DIR,
    )

    # 테스트용 데이터셋 설정
    test_dataset = VOCDataset(
        "/content/test.csv", transform=transform, img_dir=IMG_DIR, label_dir=LABEL_DIR,
    )

    # 학습 데이터 로더 설정
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=True,
    )

    # 테스트 데이터 로더 설정
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=True,
    )

    # 에폭 반복
    for epoch in range(EPOCHS):
        # for x, y in train_loader:
        #    x = x.to(DEVICE)
        #    for idx in range(8):
        #        bboxes = cellboxes_to_boxes(model(x))
        #        bboxes = non_max_suppression(bboxes[idx], iou_threshold=0.5, threshold=0.4, box_format="midpoint")
        #        plot_image(x[idx].permute(1,2,0).to("cpu"), bboxes)

        #    import sys
        #    sys.exit()


        # --- 예측 박스 및 정답 박스 수집 후 mAP 계산 ---
        pred_boxes, target_boxes = get_bboxes(
            train_loader, model, iou_threshold=0.5, threshold=0.4
        )

        mean_avg_prec = mean_average_precision(
            pred_boxes, target_boxes, iou_threshold=0.5, box_format="midpoint"
        )
        print(f"Train mAP: {mean_avg_prec}")


        # --- mAP이 기준 이상일 경우 모델 저장 ---
        # if mean_avg_prec > 0.9:
        #     checkpoint = {
        #         "state_dict": model.state_dict(),
        #         "optimizer": optimizer.state_dict(),
        #     }
        #     save_checkpoint(checkpoint, filename=LOAD_MODEL_FILE)
        #     import time
        #     time.sleep(10)

        # --- 학습 수행 ---
        train_fn(train_loader, model, optimizer, loss_fn)


if __name__ == "__main__":
    main()

Train mAP: 0.0


100%|██████████| 6/6 [00:02<00:00,  2.45it/s, loss=999]

Mean loss was 966.844980875651


Train mAP: 0.0


100%|██████████| 6/6 [00:02<00:00,  2.23it/s, loss=428]

...

100%|██████████| 6/6 [00:02<00:00,  2.46it/s, loss=55.1]

Mean loss was 37.61831474304199


Train mAP: 0.8522560000419617


100%|██████████| 6/6 [00:02<00:00,  2.26it/s, loss=39.8]

Mean loss was 38.194135665893555


In [ ]:
from PIL import Image
import torch

# 1. Load image
image = Image.open("data/images/007240.jpg").convert("RGB")

# 2. Apply transform
image_tensor, _ = transform(image, [])
image_tensor = image_tensor.unsqueeze(0).to("cuda")

# 3. Predict
model.eval()
with torch.no_grad():
    preds = model(image_tensor)

# 4. Post-process
bboxes = cellboxes_to_boxes(preds)
bboxes = non_max_suppression(bboxes[0], iou_threshold=0.5, threshold=0.1, box_format="midpoint")

# 5. Plot
plot_image(image_tensor[0].cpu().permute(1, 2, 0), bboxes)
